In [1]:
"""
================================================================================
ABLATION STUDY: CRYPTOGUARD-MLP ON D, D1, D2, ..., D8
All 5 Folds in 2 Combined Plots (Loss & Accuracy) Per Dataset
Universal Colors + Synced Fonts with Part 2B
================================================================================
"""

import os
import sys
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import Sequential, regularizers
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (matthews_corrcoef, accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix)
import matplotlib.pyplot as plt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.set_visible_devices([], 'GPU')

DATE = "14082026"
CV_FOLDS = 5
EPOCHS_FINAL = 100
PATIENCE_ES = 5
PATIENCE_LR = 5

# ══════════════════════════════════════════════════════════════════════════════
# ABLATION DATASETS
# ══════════════════════════════════════════════════════════════════════════════

DATASETS_TO_ABLATE = ['D', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8']

# Dataset display mapping
DATASET_DISPLAY_MAP = {
    'D': 'D',
    'D1': r'D\C1',
    'D2': r'D\C2',
    'D3': r'D\C3',
    'D4': r'D\C4',
    'D5': r'D\C5',
    'D6': r'D\C6',
    'D7': r'D\C7',
    'D8': r'D\C8'
}

FILE_RESULTS = "./0. D_CryptoGuard-MLP_GridSearchCV_5Fold_08082026/1.1_D_CryptoGuard-MLP_GridSearchCV_5Fold_Results_08082026.csv"

OUT_DIR = f"./2. CryptoGuard-MLP_AblationStudy_{DATE}"
os.makedirs(OUT_DIR, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# COLOR & FONT CONFIGURATION (SYNCED WITH PART 2B)
# ══════════════════════════════════════════════════════════════════════════════

FOLD_COLORS = {
    1: '#FF0000',  # Red
    2: '#FF8000',  # Orange
    3: '#006600',  # Green
    4: '#FF00FF',  # Magenta/Pink
    5: '#0000FF'   # Blue
}

FOLD_LABELS = {
    1: 'Fold #1',
    2: 'Fold #2',
    3: 'Fold #3',
    4: 'Fold #4',
    5: 'Fold #5'
}

# ══════════════════════════════════════════════════════════════════════════════
# COLOR & LINE STYLE MAPPING
# ══════════════════════════════════════════════════════════════════════════════
#
# Fold #1:
#   ├─ Train: Red solid line (─)
#   └─ Val: Red dashed line (- -)
#
# Fold #2:
#   ├─ Train: Orange solid line (─)
#   └─ Val: Orange dashed line (- -)
#
# Fold #3:
#   ├─ Train: Magenta solid line (─)
#   └─ Val: Magenta dashed line (- -)
#
# Fold #4:
#   ├─ Train: Blue solid line (─)
#   └─ Val: Blue dashed line (- -)
#
# Fold #5:
#   ├─ Train: Violet solid line (─)
#   └─ Val: Violet dashed line (- -)
#
# ══════════════════════════════════════════════════════════════════════════════

# FONT SIZES (SYNCED WITH PART 2B)
FS_TITLE = 18
FS_LABEL = 16
FS_LEGEND = 16
FS_TICK = 14

# LINE STYLES
LINE_WIDTH = 1.5
LINE_STYLE_TRAIN = '-'      # Solid (Train)
LINE_STYLE_VAL = '--'       # Dashed (Val)
ALPHA = 1.0

def print_header(title):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_step(step_num, title):
    print(f"\n[STEP {step_num}] {title}")
    print("-" * 80)

def create_mlp_model(hidden_layer1, hidden_layer2, hidden_layer3, 
                     dropout_rate, l2_reg, learning_rate, input_dim):
    """Create CryptoGuard-MLP"""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(hidden_layer1, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(hidden_layer2, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(hidden_layer3, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(1, activation='sigmoid',
              kernel_regularizer=regularizers.l2(l2_reg))
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def get_callbacks():
    """Get training callbacks"""
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                     restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                         patience=PATIENCE_LR, min_lr=1e-6, verbose=0)
    ]

def calculate_metrics(y_true, y_pred):
    """Calculate all metrics"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    
    return {
        'TP': int(tp),
        'FP': int(fp),
        'FN': int(fn),
        'TN': int(tn),
        'Accuracy': acc,
        'FPR': fpr,
        'FNR': fnr,
        'MCC': mcc,
        'Precision': prec,
        'Recall': rec,
        'F-measure': f1
    }

if __name__ == "__main__":
    
    print_header("ABLATION STUDY: CRYPTOGUARD-MLP ON D, D\\C1, D\\C2, ..., D\\C8")
    
    # STEP 1: Load Part 1 results (GridSearch best config)
    print_step(1, "LOADING PART 1 RESULTS (LOCKED MLP CONFIGURATION)")
    
    if not os.path.exists(FILE_RESULTS):
        print(f"❌ Error: Part 1 results not found")
        sys.exit(1)
    
    results_df = pd.read_csv(FILE_RESULTS)
    best = results_df.iloc[0]
    
    h1 = int(best['param_hidden_layer1'])
    h2 = int(best['param_hidden_layer2'])
    h3 = int(best['param_hidden_layer3'])
    dr = float(best['param_dropout_rate'])
    l2r = float(best['param_l2_reg'])
    lr = float(best['param_learning_rate'])
    bs = int(best['param_batch_size'])
    
    print(f"\n✓ Locked MLP Configuration:")
    print(f"  Architecture  : {h1}-{h2}-{h3}")
    print(f"  Dropout       : {dr}")
    print(f"  L2            : {l2r}")
    print(f"  LR            : {lr}")
    print(f"  Batch Size    : {bs}")
    
    # Store all ablation results for summary CSV
    all_ablation_results = []
    
    # ════════════════════════════════════════════════════════════════════════════
    # LOOP THROUGH EACH DATASET
    # ════════════════════════════════════════════════════════════════════════════
    
    for dataset_idx, DATASET in enumerate(DATASETS_TO_ABLATE):
        
        DATASET_DISPLAY = DATASET_DISPLAY_MAP[DATASET]
        
        print_header(f"DATASET {dataset_idx} / {len(DATASETS_TO_ABLATE)-1}: {DATASET_DISPLAY}")
        
        # STEP 2: Load dataset
        print_step(2, f"LOADING DATASET {DATASET_DISPLAY}")
        
        file_path = f'./Datasets/{DATASET}.xlsx'
        if not os.path.exists(file_path):
            print(f"❌ Error: Dataset file not found: {file_path}")
            sys.exit(1)
        
        df = pd.read_excel(file_path)
        X = df.drop(columns=['CLASS_LABEL'])
        y = df['CLASS_LABEL']
        input_dim = X.shape[1]
        
        print(f"✓ X shape: {X.shape}")
        print(f"✓ y shape: {y.shape}")
        print(f"✓ Input dim: {input_dim}")
        
        # STEP 3: 5-Fold CV - Collect all fold histories
        print_step(3, f"5-FOLD STRATIFIED CV ON {DATASET_DISPLAY}")
        
        skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
        fold_metrics_list = []
        fold_histories = {}
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            print(f"\n  [Fold {fold}/{CV_FOLDS}]")
            
            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]
            
            X_train_real, X_val, y_train_real, y_val = train_test_split(
                X_train, y_train,
                test_size=0.10,
                random_state=SEED,
                stratify=y_train
            )
            
            scaler = StandardScaler()
            X_train_real_sc = scaler.fit_transform(X_train_real).astype(np.float32)
            X_val_sc = scaler.transform(X_val).astype(np.float32)
            X_test_sc = scaler.transform(X_test).astype(np.float32)
            y_train_real = y_train_real.values.astype(np.float32)
            y_val = y_val.values.astype(np.float32)
            y_test = y_test.values.astype(np.float32)
            
            model = create_mlp_model(h1, h2, h3, dr, l2r, lr, input_dim)
            
            history = model.fit(
                X_train_real_sc, y_train_real,
                epochs=EPOCHS_FINAL,
                batch_size=bs,
                validation_data=(X_val_sc, y_val),
                callbacks=get_callbacks(),
                verbose=0
            )
            
            epochs_trained = len(history.history['loss'])
            
            fold_histories[fold] = {
                'epochs': np.arange(1, epochs_trained + 1),
                'train_loss': history.history['loss'],
                'val_loss': history.history['val_loss'],
                'train_acc': history.history['accuracy'],
                'val_acc': history.history['val_accuracy']
            }
            
            y_pred_proba = model.predict(X_test_sc, verbose=0, batch_size=bs)
            y_pred = (y_pred_proba > 0.5).astype(int).flatten()
            
            metrics = calculate_metrics(y_test, y_pred)
            
            fold_metrics_list.append({
                'Fold': f'Fold {fold}',
                'Epochs_Trained': epochs_trained,
                'TP': metrics['TP'],
                'FP': metrics['FP'],
                'FN': metrics['FN'],
                'TN': metrics['TN'],
                'Accuracy': metrics['Accuracy'],
                'FPR': metrics['FPR'],
                'FNR': metrics['FNR'],
                'MCC': metrics['MCC'],
                'Precision': metrics['Precision'],
                'Recall': metrics['Recall'],
                'F-measure': metrics['F-measure'],
                'Best_H1': h1,
                'Best_H2': h2,
                'Best_H3': h3,
                'Best_Dropout': dr,
                'Best_L2': l2r,
                'Best_LR': lr,
                'Best_BatchSize': bs
            })
            
            print(f"    Acc={metrics['Accuracy']:.4f}, MCC={metrics['MCC']:.4f}, Epochs={epochs_trained}")
            
            del model, X_train, X_test, X_train_real, X_val, X_train_real_sc, X_val_sc, X_test_sc, history
            tf.keras.backend.clear_session()
            gc.collect()
        
        # STEP 4: Calculate mean & std
        fold_metrics_df = pd.DataFrame(fold_metrics_list)
        
        mean_row = {
            'Fold': 'Mean',
            'Epochs_Trained': '-',
            'TP': '-',
            'FP': '-',
            'FN': '-',
            'TN': '-',
            'Accuracy': f"{fold_metrics_df['Accuracy'].mean():.4f}±{fold_metrics_df['Accuracy'].std():.4f}",
            'FPR': f"{fold_metrics_df['FPR'].mean():.4f}±{fold_metrics_df['FPR'].std():.4f}",
            'FNR': f"{fold_metrics_df['FNR'].mean():.4f}±{fold_metrics_df['FNR'].std():.4f}",
            'MCC': f"{fold_metrics_df['MCC'].mean():.4f}±{fold_metrics_df['MCC'].std():.4f}",
            'Precision': f"{fold_metrics_df['Precision'].mean():.4f}±{fold_metrics_df['Precision'].std():.4f}",
            'Recall': f"{fold_metrics_df['Recall'].mean():.4f}±{fold_metrics_df['Recall'].std():.4f}",
            'F-measure': f"{fold_metrics_df['F-measure'].mean():.4f}±{fold_metrics_df['F-measure'].std():.4f}",
            'Best_H1': '',
            'Best_H2': '',
            'Best_H3': '',
            'Best_Dropout': '',
            'Best_L2': '',
            'Best_LR': '',
            'Best_BatchSize': ''
        }
        
        final_df = pd.concat([fold_metrics_df, pd.DataFrame([mean_row])], ignore_index=True)
        
        # STEP 5: Save CSV for this dataset
        file_csv = f"{OUT_DIR}/{dataset_idx}.3_{DATASET}_CryptoGuard-MLP_AblationStudy_Evaluation_{DATE}.csv"
        final_df.to_csv(file_csv, index=False)
        print(f"\n✓ Saved CSV: {os.path.basename(file_csv)}")
        
        # Store mean results for summary
        all_ablation_results.append({
            'Dataset': DATASET_DISPLAY,
            'Input_Dim': input_dim,
            'Accuracy': f"{fold_metrics_df['Accuracy'].mean():.4f}±{fold_metrics_df['Accuracy'].std():.4f}",
            'FPR': f"{fold_metrics_df['FPR'].mean():.4f}±{fold_metrics_df['FPR'].std():.4f}",
            'FNR': f"{fold_metrics_df['FNR'].mean():.4f}±{fold_metrics_df['FNR'].std():.4f}",
            'MCC': f"{fold_metrics_df['MCC'].mean():.4f}±{fold_metrics_df['MCC'].std():.4f}",
            'Precision': f"{fold_metrics_df['Precision'].mean():.4f}±{fold_metrics_df['Precision'].std():.4f}",
            'Recall': f"{fold_metrics_df['Recall'].mean():.4f}±{fold_metrics_df['Recall'].std():.4f}",
            'F-measure': f"{fold_metrics_df['F-measure'].mean():.4f}±{fold_metrics_df['F-measure'].std():.4f}"
        })
        
        # STEP 6: Generate Plots for this dataset
        print_step(6, f"GENERATING PLOTS FOR {DATASET_DISPLAY}")
        
        # ────────────────────────────────────────────────────────────────────────
        # PLOT 1: ALL FOLDS - LOSS
        # ────────────────────────────────────────────────────────────────────────
        
        print("  Generating Loss plot...")
        fig, ax = plt.subplots(figsize=(14, 7))
        
        for fold in range(1, CV_FOLDS + 1):
            epochs = fold_histories[fold]['epochs']
            train_loss = fold_histories[fold]['train_loss']
            val_loss = fold_histories[fold]['val_loss']
            color = FOLD_COLORS[fold]
            label = FOLD_LABELS[fold]
            
            ax.plot(epochs, train_loss, color=color, linewidth=LINE_WIDTH, 
                    linestyle=LINE_STYLE_TRAIN, alpha=ALPHA, label=label)
            ax.plot(epochs, val_loss, color=color, linewidth=LINE_WIDTH, 
                    linestyle=LINE_STYLE_VAL, alpha=ALPHA)
        
        ax.plot([], [], color='black', linewidth=LINE_WIDTH, 
                linestyle=LINE_STYLE_TRAIN, label='Train')
        ax.plot([], [], color='black', linewidth=LINE_WIDTH, 
                linestyle=LINE_STYLE_VAL, label='Val')
        
        ax.set_title(f'CryptoGuard-MLP - Training and Validation Loss (5-Fold CV) on {DATASET_DISPLAY}', 
                    fontsize=FS_TITLE, pad=15)
        ax.set_xlabel('Epoch', fontsize=FS_LABEL)
        ax.set_ylabel('Loss', fontsize=FS_LABEL)
        ax.tick_params(axis='both', labelsize=FS_TICK)
        ax.legend(fontsize=FS_LEGEND, loc='upper right', frameon=True, shadow=True, ncol=1)
        ax.grid(True, alpha=0.3, linestyle=':')
        
        plt.tight_layout()
        file_loss = f"{OUT_DIR}/{dataset_idx}.1_{DATASET}_CryptoGuard-MLP_AblationStudy_Loss_{DATE}.png"
        plt.savefig(file_loss, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  ✓ Saved: {os.path.basename(file_loss)}")
        
        # ────────────────────────────────────────────────────────────────────────
        # PLOT 2: ALL FOLDS - ACCURACY
        # ────────────────────────────────────────────────────────────────────────
        
        print("  Generating Accuracy plot...")
        fig, ax = plt.subplots(figsize=(14, 7))
        
        for fold in range(1, CV_FOLDS + 1):
            epochs = fold_histories[fold]['epochs']
            train_acc = fold_histories[fold]['train_acc']
            val_acc = fold_histories[fold]['val_acc']
            color = FOLD_COLORS[fold]
            label = FOLD_LABELS[fold]
            
            ax.plot(epochs, train_acc, color=color, linewidth=LINE_WIDTH, 
                    linestyle=LINE_STYLE_TRAIN, alpha=ALPHA, label=label)
            ax.plot(epochs, val_acc, color=color, linewidth=LINE_WIDTH, 
                    linestyle=LINE_STYLE_VAL, alpha=ALPHA)
        
        ax.plot([], [], color='black', linewidth=LINE_WIDTH, 
                linestyle=LINE_STYLE_TRAIN, label='Train')
        ax.plot([], [], color='black', linewidth=LINE_WIDTH, 
                linestyle=LINE_STYLE_VAL, label='Val')
        
        ax.set_title(f'CryptoGuard-MLP - Training and Validation Accuracy (5-Fold CV) on {DATASET_DISPLAY}', 
                    fontsize=FS_TITLE, pad=15)
        ax.set_xlabel('Epoch', fontsize=FS_LABEL)
        ax.set_ylabel('Accuracy', fontsize=FS_LABEL)
        ax.tick_params(axis='both', labelsize=FS_TICK)
        ax.legend(fontsize=FS_LEGEND, loc='lower right', frameon=True, shadow=True, ncol=1)
        ax.grid(True, alpha=0.3, linestyle=':')
        
        plt.tight_layout()
        file_acc = f"{OUT_DIR}/{dataset_idx}.2_{DATASET}_CryptoGuard-MLP_AblationStudy_Accuracy_{DATE}.png"
        plt.savefig(file_acc, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  ✓ Saved: {os.path.basename(file_acc)}")
    
    # ════════════════════════════════════════════════════════════════════════════
    # STEP 7: Generate Summary CSV (All Datasets)
    # ════════════════════════════════════════════════════════════════════════════
    
    print_header("STEP 7: GENERATING ABLATION SUMMARY CSV")
    
    summary_df = pd.DataFrame(all_ablation_results)
    file_summary = f"{OUT_DIR}/2. CryptoGuard-MLP_AblationStudy_Evaluation_Summary_{DATE}.csv"
    summary_df.to_csv(file_summary, index=False)
    
    print(f"✓ Saved Summary: {os.path.basename(file_summary)}")
    print(f"\nSummary CSV Structure:")
    print(f"  Rows: D, D\\C1, D\\C2, ..., D\\C8")
    print(f"  Columns: Dataset, Input_Dim, Accuracy, FPR, FNR, MCC, Precision, Recall, F-measure")
    
    print_header("✅ ABLATION STUDY COMPLETE!")
    print(f"\nOutput Directory: {OUT_DIR}")
    print(f"\nGenerated Files Structure:")
    print(f"  For each dataset (D, D\\C1):")
    print(f"    - 0.1_D_CryptoGuard-MLP_AblationStudy_Loss_{DATE}.png")
    print(f"    - 0.2_D_CryptoGuard-MLP_AblationStudy_Accuracy_{DATE}.png")
    print(f"    - 0.3_D_CryptoGuard-MLP_AblationStudy_Evaluation_{DATE}.csv")
    print(f"    - 1.1_D1_CryptoGuard-MLP_AblationStudy_Loss_{DATE}.png")
    print(f"    - 1.2_D1_CryptoGuard-MLP_AblationStudy_Accuracy_{DATE}.png")
    print(f"    - 1.3_D1_CryptoGuard-MLP_AblationStudy_Evaluation_{DATE}.csv")
    print(f"  Summary:")
    print(f"    - 2. CryptoGuard-MLP_AblationStudy_Evaluation_Summary_{DATE}.csv")
    print(f"\nConfiguration (Synced with Part 2B):")
    print(f"  Line Width: {LINE_WIDTH}")
    print(f"  Alpha: {ALPHA}")
    print(f"  Font Sizes: Title={FS_TITLE}, Label={FS_LABEL}, Legend={FS_LEGEND}, Tick={FS_TICK}")
    print_header("END OF ABLATION STUDY")


ABLATION STUDY: CRYPTOGUARD-MLP ON D, D\C1, D\C2, ..., D\C8

[STEP 1] LOADING PART 1 RESULTS (LOCKED MLP CONFIGURATION)
--------------------------------------------------------------------------------

✓ Locked MLP Configuration:
  Architecture  : 768-512-128
  Dropout       : 0.5
  L2            : 0.005
  LR            : 0.0001
  Batch Size    : 64

DATASET 0 / 8: D

[STEP 2] LOADING DATASET D
--------------------------------------------------------------------------------
✓ X shape: (20000, 525)
✓ y shape: (20000,)
✓ Input dim: 525

[STEP 3] 5-FOLD STRATIFIED CV ON D
--------------------------------------------------------------------------------

  [Fold 1/5]
    Acc=0.9303, MCC=0.8606, Epochs=65

  [Fold 2/5]
    Acc=0.9313, MCC=0.8625, Epochs=69

  [Fold 3/5]
    Acc=0.9273, MCC=0.8546, Epochs=50

  [Fold 4/5]
    Acc=0.9363, MCC=0.8725, Epochs=63

  [Fold 5/5]
    Acc=0.9277, MCC=0.8560, Epochs=63

✓ Saved CSV: 0.3_D_CryptoGuard-MLP_AblationStudy_Evaluation_14082026.csv

[STEP 6